In [1]:

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


data = {
    'X1': [4, 2, 2, 3, 4, 9, 6, 9, 8, 10],
    'X2': [2, 4, 3, 6, 4, 10, 8, 5, 7, 8],
    'Class': [0, 0, 0, 0, 0, 1, 1, 1, 1, 1]
}
df = pd.DataFrame(data)




X = df[['X1', 'X2']].values
y = df['Class'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


pca = PCA()
pca.fit(X_scaled)


max_variance_component_index = np.argmax(pca.explained_variance_ratio_)
max_variance_component = pca.components_[max_variance_component_index]
explained_variance_ratio = pca.explained_variance_ratio_[max_variance_component_index]
explained_variance = pca.explained_variance_[max_variance_component_index]



print("RESULTS:")
print(f"\nPrincipal Component with Maximum Variance Ratio: PC{max_variance_component_index + 1}")
print(f"Component coefficients (weights): {max_variance_component}")
print(f"Explained Variance Ratio: {explained_variance_ratio:.4f} ({explained_variance_ratio*100:.2f}%)")
print(f"Explained Variance: {explained_variance:.4f}")



print("Explained Variance Ratio for ALL Components:")
for i, ratio in enumerate(pca.explained_variance_ratio_):
    print(f"PC{i+1}: {ratio:.4f} ({ratio*100:.2f}%)")



RESULTS:

Principal Component with Maximum Variance Ratio: PC1
Component coefficients (weights): [0.70710678 0.70710678]
Explained Variance Ratio: 0.8542 (85.42%)
Explained Variance: 1.8982
Explained Variance Ratio for ALL Components:
PC1: 0.8542 (85.42%)
PC2: 0.1458 (14.58%)


In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.metrics import accuracy_score


df = pd.read_csv("cancer (1) (2).csv")
X = df.drop(columns=["Id", "Diagnosis"])
y = df["Diagnosis"]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train = StandardScaler().fit_transform(X_train)
X_test  = StandardScaler().fit_transform(X_test)


def score(X_tr, X_te, name):
    model = LogisticRegression(max_iter=10000)
    model.fit(X_tr, y_train)
    acc = accuracy_score(y_test, model.predict(X_te))
    return {"Method": name, "Accuracy": round(acc, 4), "Features": X_tr.shape[1]}


full = score(X_train, X_test, "Full (30 features)")

pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train)
X_test_pca  = pca.transform(X_test)
pca_res = score(X_train_pca, X_test_pca, f"PCA ({pca.n_components_} components)")

lda = LDA(n_components=1)
X_train_lda = lda.fit_transform(X_train, y_train)
X_test_lda  = lda.transform(X_test)
lda_res = score(X_train_lda, X_test_lda, "LDA (1 component)")

print(pd.DataFrame([full, pca_res, lda_res]).to_string(index=False))



             Method  Accuracy  Features
 Full (30 features)    0.9825        30
PCA (10 components)    0.9825        10
  LDA (1 component)    0.9298         1


In [7]:
# COMPLETE & CORRECT CODE — Breast Cancer Classification Comparison
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.metrics import accuracy_score

# 1. Load data
df = pd.read_csv("cancer (1) (2).csv")

# 2. Features and target
X = df.drop(columns=["Id", "Diagnosis"])
y = df["Diagnosis"]  # M = Malignant, B = Benign

# 3. Train-test split (stratified for balanced classes)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. CORRECT SCALING — ONE scaler only!
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # Fit on train
X_test_scaled = scaler.transform(X_test)         # Use same scaler on test ← CRITICAL!

# 5. Function to train Logistic Regression and return results
def evaluate(X_tr, X_te, method_name):
    model = LogisticRegression(max_iter=10000, random_state=42)
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    accuracy = accuracy_score(y_test, y_pred)
    return {
        "Method": method_name,
        "Accuracy": round(accuracy, 4),
        "Features Used": X_tr.shape[1]
    }

# 6. Results list
results = []

# Full model (30 features)
results.append(evaluate(X_train_scaled, X_test_scaled, "Full Data (30 features)"))

# PCA: Keep 95% of variance
pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)
results.append(evaluate(X_train_pca, X_test_pca, f"PCA (95% variance → {pca.n_components_} components)"))

# LDA: Best for 2-class problems → only 1 component possible
lda = LDA(n_components=1)
X_train_lda = lda.fit_transform(X_train_scaled, y_train)
X_test_lda = lda.transform(X_test_scaled)
results.append(evaluate(X_train_lda, X_test_lda, "LDA (1 component - Supervised)"))

# 7. Final beautiful table
print("\n" + "="*60)
print("     BREAST CANCER CLASSIFICATION RESULTS")
print("="*60)
print(pd.DataFrame(results).to_string(index=False))
print("="*60)


     BREAST CANCER CLASSIFICATION RESULTS
                            Method  Accuracy  Features Used
           Full Data (30 features)    0.9649             30
PCA (95% variance → 10 components)    0.9737             10
    LDA (1 component - Supervised)    0.9737              1
